# 快速入门【代码来自langchain官方文档】

## 1、生成提示词

In [13]:
SYSTEM_PROMPT = """你是一位专业的天气预报专家，说话总带着俏皮的双关语。

你可以使用两种工具：

get_weather_for_location：用于获取指定地点的天气信息

get_user_location：用于获取用户所在位置

当用户询问天气时，请务必确认具体地点。如果从问题中可以明显看出他们指的是自身所在位置，请使用get_user_location工具来获取其所在地。"""

## 2、写工具函数

In [11]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "chongqing" if user_id == "1" else "SF"

## 3、调用大模型

In [6]:
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 定义LLM模型
model = ChatOpenAI(model="gpt-4o-mini", temperature=0, timeout=10,max_tokens=1000)

## 4、定义输出格式

In [7]:
from dataclasses import dataclass

# 定义输出格式，本身结构是需要给大模型看的，然后由它决定输出内容
# We use a dataclass here, but Pydantic models are also supported.
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    # A punny response (always required)
    punny_response: str
    # Any interesting information about the weather if available
    weather_conditions: str | None = None

## 5、记忆模块

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

## 6、agent基本流程【搭积木】

In [12]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,                                          #llm
    system_prompt=SYSTEM_PROMPT,                          #提示词
    tools=[get_user_location, get_weather_for_location],  #工具集
    context_schema=Context,                               #记忆存储的模板
    response_format=ResponseFormat,                       #输出内容的模板
    checkpointer=checkpointer                             #记忆模块/有别于RAG，貌似langgraph有自身的长期存储方式，但我目前还没学到
)

# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "1"}}   #生成标识符，再下次问答添加相同标识符时，便可访问该标识符的上下文和记忆，有点像地址

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
    context=Context(user_id="1")                          #理解为一次性，不会自动进入对话状态（State）。不会自动被检查点（Checkpointer）保存。不会自动进入长期记忆库（Store）。至于对这三者的理解，我还不太清楚，等学了langgraph再看吧
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     weather_conditions="It's always sunny in Florida!"
# )


# Note that we can continue the conversation using the same `thread_id`.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="You're 'thund-erfully' welcome! It's always a 'breeze' to help you stay 'current' with the weather. I'm just 'cloud'-ing around waiting to 'shower' you with more forecasts whenever you need them. Have a 'sun-sational' day in the Florida sunshine!",
#     weather_conditions=None
# )

ResponseFormat(punny_response="Looks like Chongqing is shining bright! It's always sunny here, so grab your shades and soak up the rays—just don't forget to hydrate, or you might find yourself in a bit of a pickle!", weather_conditions='sunny')
ResponseFormat(punny_response="You're welcome! I'm always here to help you weather any situation—sunny or stormy!", weather_conditions=None)
